# CS 435 — Project 3: Medical X-Ray Anomaly Detection
## Phase 2: CNN Baseline (No GAN Augmentation)

**Goal of this notebook:**  
Build, train, and evaluate a Convolutional Neural Network (CNN) on the real NIH Chest X-Ray14 data from Phase 1 — with NO synthetic augmentation. This is our **control experiment**. The metrics we record here (AUC-ROC, F1, accuracy) become the baseline we will compare against in Phase 5, after the GAN augmentation from Phase 3.

**Why binary crossentropy with sigmoid output?**  
This is a **multi-label** classification problem — a single X-ray can show multiple diseases simultaneously. As Chollet explains in *Deep Learning with Python* (Ch. 14): for multi-label classification, use sigmoid activations on the output layer (one independent probability per class) and binary crossentropy as the loss. This is different from softmax + categorical crossentropy, which forces exactly one class to win.

**Prerequisites:** Phase 1 notebook must have been run and splits saved to Google Drive.

---
## Step 1: Mount Drive and import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay
)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Step 2: Configuration — all hyperparameters in one place

Keeping every tunable setting at the top of the notebook means you only need to change values here, not hunt through code. This is a best practice for reproducible experiments.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath("")
SPLITS_DIR  = os.path.join(BASE_DIR, 'splits')
MODEL_DIR   = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Image settings (must match Phase 1) ───────────────────────────────────────
IMG_SIZE   = 224          # height and width in pixels
CHANNELS   = 1            # grayscale X-rays
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)

# ── Label settings (must match Phase 1) ───────────────────────────────────────
TARGET_CLASSES = ['No Finding', 'Atelectasis', 'Effusion', 'Cardiomegaly', 'Pneumonia']
NUM_CLASSES    = len(TARGET_CLASSES)   # 5

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 32
EPOCHS        = 30
LEARNING_RATE = 1e-3
PATIENCE      = 5
THRESHOLD     = 0.5

print("Configuration loaded.")
print(f"  Input shape : {INPUT_SHAPE}")
print(f"  Classes     : {TARGET_CLASSES}")
print(f"  Batch size  : {BATCH_SIZE}")

---
## Step 3: Reload the train / val / test splits from Phase 1

In [ ]:
train_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(SPLITS_DIR, 'val.csv'))
test_df  = pd.read_csv(os.path.join(SPLITS_DIR, 'test.csv'))

print(f"Train : {len(train_df):,} images")
print(f"Val   : {len(val_df):,} images")
print(f"Test  : {len(test_df):,} images")

---
## Step 4: Rebuild the tf.data pipelines

We replicate the Phase 1 preprocessing function exactly so the data feeding
the CNN is identical to what the GAN will later be trained on.

In [ ]:
def preprocess_image(filepath, label):
    """
    Load → decode (grayscale) → resize (224×224) → normalize [0,1].
    Identical to Phase 1 to ensure consistency.
    """
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_png(raw, channels=CHANNELS)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def build_dataset(dataframe, shuffle=False):
    filepaths = dataframe['filepath'].values
    labels    = dataframe[TARGET_CLASSES].values.astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_dataset = build_dataset(train_df, shuffle=True)
val_dataset   = build_dataset(val_df,   shuffle=False)
test_dataset  = build_dataset(test_df,  shuffle=False)

print(f"Train batches : {len(train_dataset)}")
print(f"Val batches   : {len(val_dataset)}")
print(f"Test batches  : {len(test_dataset)}")

---
## Step 5: Build the CNN architecture

### Architecture decisions explained:

**Block structure:** Three convolutional blocks, each with:
- `Conv2D` — learns spatial features (edges, textures, shapes) via learned filters
- `BatchNormalization` — normalizes activations to keep training stable and fast. Chollet (*Deep Learning with Python*, Ch. 9) recommends placing BN *before* the activation
- `ReLU activation` — introduces non-linearity; kills negative activations to zero
- `MaxPooling2D` — downsamples the spatial dimensions by half, reducing computation and giving the network translational invariance

**Filter progression (32 → 64 → 128):** We double the number of filters each block. As spatial size shrinks (224→112→56→28), filter depth increases so the network can learn richer, more abstract features — a standard CNN design pattern.

**GlobalAveragePooling2D instead of Flatten:** GAP reduces each feature map to a single number (its average). This drastically reduces parameters compared to Flatten and provides built-in regularization against overfitting — a technique highlighted in both Chollet and Rosebrock.

**Output layer:** `Dense(NUM_CLASSES, activation='sigmoid')` — one sigmoid neuron per class. Each neuron independently outputs a probability in [0, 1]. This is the correct choice for multi-label classification (as opposed to softmax, which forces probabilities to sum to 1 and implies mutual exclusivity).

**Loss:** `binary_crossentropy` — evaluates each class independently as a binary (positive/negative) decision. Perfect for multi-label problems.

In [ ]:
def build_cnn(input_shape, num_classes):
    """
    Build the CNN baseline model using the Keras Functional API.

    Architecture:
        Input (224, 224, 1)
        └─ Conv Block 1: Conv2D(32) → BN → ReLU → MaxPool → Dropout(0.25)
        └─ Conv Block 2: Conv2D(64) → BN → ReLU → MaxPool → Dropout(0.25)
        └─ Conv Block 3: Conv2D(128) → BN → ReLU → MaxPool → Dropout(0.25)
        └─ GlobalAveragePooling2D
        └─ Dense(256) → BN → ReLU → Dropout(0.5)
        └─ Dense(num_classes, sigmoid)   ← multi-label output
    """
    inputs = keras.Input(shape=input_shape, name='xray_input')

    # ── Convolutional Block 1 ──────────────────────────────────────────────
    # Input: 224×224×1  →  Output: 112×112×32
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False,
                      name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = layers.Dropout(0.25, name='drop1')(x)

    # ── Convolutional Block 2 ──────────────────────────────────────────────
    # Input: 112×112×32  →  Output: 56×56×64
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False,
                      name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = layers.Dropout(0.25, name='drop2')(x)

    # ── Convolutional Block 3 ──────────────────────────────────────────────
    # Input: 56×56×64  →  Output: 28×28×128
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False,
                      name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    x = layers.Dropout(0.25, name='drop3')(x)

    # ── Global Average Pooling ─────────────────────────────────────────────
    # Collapses each 28×28 feature map to a single number → (batch, 128)
    x = layers.GlobalAveragePooling2D(name='gap')(x)

    # ── Fully Connected Head ───────────────────────────────────────────────
    x = layers.Dense(256, use_bias=False, name='fc1')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.Activation('relu', name='relu4')(x)
    x = layers.Dropout(0.5, name='drop4')(x)   # higher dropout in dense layer

    # ── Output Layer ──────────────────────────────────────────────────────
    # sigmoid: each of the 5 outputs is an independent probability in [0, 1]
    # This is correct for MULTI-LABEL classification (not softmax)
    outputs = layers.Dense(num_classes, activation='sigmoid',
                           name='predictions')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='xray_cnn_baseline')
    return model


# Build and summarize
model = build_cnn(INPUT_SHAPE, NUM_CLASSES)
model.summary()

---
## Step 6: Compile the model

### Why these choices:
- **Adam optimizer:** Adaptive learning rate — handles the sparse gradients caused by class imbalance better than plain SGD
- **binary_crossentropy:** Treats each class as its own binary problem, as recommended by Chollet for multi-label tasks
- **AUC metric:** Area Under the ROC Curve — the gold standard for imbalanced medical classification problems. Accuracy is misleading when one class dominates (a model that always predicts 'No Finding' can get 60%+ accuracy while being clinically useless)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy', threshold=THRESHOLD),
        keras.metrics.AUC(name='auc', multi_label=True),          # macro AUC across all 5 classes
        keras.metrics.Precision(name='precision', thresholds=THRESHOLD),
        keras.metrics.Recall(name='recall',    thresholds=THRESHOLD),
    ]
)

print("Model compiled.")
print(f"  Optimizer : Adam (lr={LEARNING_RATE})")
print(f"  Loss      : binary_crossentropy")
print(f"  Metrics   : accuracy, AUC, precision, recall")

---
## Step 7: Set up training callbacks

Callbacks are objects that run at specific points during training to monitor and control the process (Chollet, *Deep Learning with Python*, Ch. 7).

We use three:
1. **`ModelCheckpoint`** — saves the best model weights seen so far (best = lowest `val_loss`). This means even if training continues for more epochs after the best point, we always have the best weights saved to disk.
2. **`EarlyStopping`** — stops training automatically when `val_loss` has not improved for `PATIENCE` consecutive epochs. This prevents wasted compute and overfitting.
3. **`ReduceLROnPlateau`** — if `val_loss` plateaus for 3 epochs, cuts the learning rate by half. This lets the optimizer take smaller steps to escape flat regions.

In [ ]:
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'baseline_cnn_best.keras')

callbacks = [
    # Save best model weights to Drive
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    # Stop training when val_loss stops improving
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,  # restore weights from best epoch on stop
        verbose=1
    ),
    # Reduce learning rate when training stalls
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # multiply LR by 0.5
        patience=3,       # wait 3 epochs before reducing
        min_lr=1e-6,      # never go below this
        verbose=1
    ),
    # Log all metrics to CSV for later analysis
    keras.callbacks.CSVLogger(
        os.path.join(RESULTS_DIR, 'baseline_training_log.csv')
    )
]

print(f"Callbacks configured. Best model will save to:\n  {CHECKPOINT_PATH}")

---
## Step 8: Train the model

This is where the CNN actually learns. Each epoch:
1. Forward pass — images flow through the network, predictions are made
2. Loss computed — binary crossentropy measures how wrong the predictions are
3. Backward pass — gradients flow back through all layers
4. Weights updated — Adam adjusts each weight to reduce the loss

Expected training time on a free Colab GPU (T4): ~3–5 minutes per epoch.

In [ ]:
print("Starting training...")
print(f"Max epochs: {EPOCHS} (EarlyStopping may terminate early)")
print("-" * 60)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete.")
print(f"Trained for {len(history.history['loss'])} epochs.")

---
## Step 9: Plot training curves

Training curves are the first diagnostic to check after training. We look for:
- **Loss decreasing** on both train and val → model is learning
- **Train loss < val loss** → some overfitting (expected and acceptable)
- **Large gap between train and val** → too much overfitting, need more regularization
- **Val loss increasing while train loss still decreasing** → classic overfit signature

In [ ]:
def plot_training_history(history, save_path=None):
    """Plot loss and AUC curves for train and validation sets."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(history.history['loss']) + 1)

    # ── Loss ──────────────────────────────────────────────────────────────
    axes[0].plot(epochs_range, history.history['loss'],     label='Train loss',      color='#2196F3')
    axes[0].plot(epochs_range, history.history['val_loss'], label='Validation loss', color='#F44336', linestyle='--')
    axes[0].set_title('Binary crossentropy loss', fontsize=12)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # ── AUC ───────────────────────────────────────────────────────────────
    axes[1].plot(epochs_range, history.history['auc'],     label='Train AUC',      color='#4CAF50')
    axes[1].plot(epochs_range, history.history['val_auc'], label='Validation AUC', color='#FF9800', linestyle='--')
    axes[1].set_title('AUC-ROC', fontsize=12)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('AUC')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('CNN Baseline — Training History (No GAN Augmentation)', fontsize=13, y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


plot_training_history(
    history,
    save_path=os.path.join(RESULTS_DIR, 'baseline_training_curves.png')
)

---
## Step 10: Evaluate on the test set

The test set has never been seen during training or validation. This gives us an honest, unbiased estimate of how the model will perform on new, real-world X-rays.

We first load the best saved checkpoint (the weights from the epoch with lowest val_loss) before evaluating.

In [ ]:
# Load the best checkpoint saved by ModelCheckpoint
best_model = keras.models.load_model(CHECKPOINT_PATH)
print(f"Loaded best model from: {CHECKPOINT_PATH}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = best_model.evaluate(test_dataset, verbose=1)
metric_names = best_model.metrics_names

print("\n=== Test Set Results ===")
for name, value in zip(metric_names, test_results):
    print(f"  {name:15s}: {value:.4f}")

---
## Step 11: Generate predictions and compute detailed metrics

Keras's `.evaluate()` gives aggregate metrics. For the project report we need per-class metrics and ROC curves, which require extracting raw predictions first.

In [ ]:
# ── Collect all predictions and true labels from test set ─────────────────
y_true_list  = []
y_prob_list  = []   # raw sigmoid probabilities

for images, labels in test_dataset:
    probs = best_model(images, training=False)   # no dropout during inference
    y_true_list.append(labels.numpy())
    y_prob_list.append(probs.numpy())

y_true = np.vstack(y_true_list)   # shape: (N_test, 5)
y_prob = np.vstack(y_prob_list)   # shape: (N_test, 5)

# Apply threshold to get binary predictions
y_pred = (y_prob >= THRESHOLD).astype(int)  # shape: (N_test, 5)

print(f"Test set predictions collected.")
print(f"  y_true shape : {y_true.shape}")
print(f"  y_prob shape : {y_prob.shape}")
print(f"  y_pred shape : {y_pred.shape}")

In [ ]:
# ── Per-class AUC-ROC ─────────────────────────────────────────────────────
print("=== Per-Class AUC-ROC (Baseline CNN — No GAN) ===")
auc_scores = {}
for i, cls in enumerate(TARGET_CLASSES):
    auc = roc_auc_score(y_true[:, i], y_prob[:, i])
    auc_scores[cls] = auc
    print(f"  {cls:20s}: AUC = {auc:.4f}")

macro_auc = np.mean(list(auc_scores.values()))
print(f"\n  {'Macro Average':20s}: AUC = {macro_auc:.4f}")

# ── Per-class F1, Precision, Recall ──────────────────────────────────────
print("\n=== Per-Class F1, Precision, Recall ===")
report = classification_report(
    y_true, y_pred,
    target_names=TARGET_CLASSES,
    zero_division=0
)
print(report)

---
## Step 12: Plot ROC curves per class

The ROC curve plots True Positive Rate vs False Positive Rate at every possible threshold. A perfect classifier has AUC = 1.0; a random classifier has AUC = 0.5. The larger the area under the curve, the better the model at discriminating positive from negative cases — especially important for imbalanced medical data.

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 4))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, (cls, ax, color) in enumerate(zip(TARGET_CLASSES, axes, colors)):
    fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
    auc_val = auc_scores[cls]

    ax.plot(fpr, tpr, color=color, lw=2, label=f'AUC = {auc_val:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
    ax.set_title(cls, fontsize=10)
    ax.set_xlabel('FPR', fontsize=9)
    ax.set_ylabel('TPR', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('ROC Curves — CNN Baseline (No GAN Augmentation)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'baseline_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 13: Visualize predictions on sample images

Always sanity-check your model visually. Seeing what the model actually predicted on real images helps catch silent failures that summary metrics can miss.

In [ ]:
# Pull one batch for visualization
sample_images, sample_labels = next(iter(test_dataset))
sample_probs = best_model(sample_images, training=False).numpy()
sample_preds = (sample_probs >= THRESHOLD).astype(int)
sample_true  = sample_labels.numpy().astype(int)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    img = sample_images[i].numpy().squeeze()

    true_classes = [TARGET_CLASSES[j] for j, v in enumerate(sample_true[i])  if v == 1]
    pred_classes = [TARGET_CLASSES[j] for j, v in enumerate(sample_preds[i]) if v == 1]

    true_str = ', '.join(true_classes) if true_classes else 'No Finding'
    pred_str = ', '.join(pred_classes) if pred_classes else 'No Finding'
    correct   = set(true_classes) == set(pred_classes)

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(
        f"True : {true_str}\nPred : {pred_str}",
        fontsize=8,
        color='green' if correct else 'red'
    )
    axes[i].axis('off')

plt.suptitle('Sample Predictions — CNN Baseline (green = correct, red = incorrect)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'baseline_sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 14: Save baseline metrics to CSV

We save all metrics in a structured CSV so Phase 5 can load both the baseline and the GAN-augmented results side by side for comparison. This is the key payoff of the project.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score as sk_f1

# Build a per-class metrics DataFrame
metrics_rows = []
for i, cls in enumerate(TARGET_CLASSES):
    metrics_rows.append({
        'class':     cls,
        'auc_roc':   roc_auc_score(y_true[:, i], y_prob[:, i]),
        'f1':        sk_f1(y_true[:, i], y_pred[:, i], zero_division=0),
        'precision': precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
        'recall':    recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
        'model':     'baseline_cnn'   # label for Phase 5 comparison
    })

# Add macro averages
metrics_rows.append({
    'class':     'MACRO_AVG',
    'auc_roc':   macro_auc,
    'f1':        sk_f1(y_true, y_pred, average='macro', zero_division=0),
    'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
    'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
    'model':     'baseline_cnn'
})

metrics_df = pd.DataFrame(metrics_rows)

# Save to Drive
metrics_path = os.path.join(RESULTS_DIR, 'baseline_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)

print("Baseline metrics saved to:", metrics_path)
print("\n=== Baseline Metrics Summary ===")
print(metrics_df.to_string(index=False))

---
## Phase 2 Summary

| Step | What we did | Output |
|------|-------------|--------|
| 1–2  | Environment setup + hyperparameter config | Ready to train |
| 3–4  | Reloaded Phase 1 splits + rebuilt tf.data pipeline | `train/val/test_dataset` |
| 5    | Built CNN: 3 Conv blocks → GAP → Dense → sigmoid | Model architecture |
| 6    | Compiled with Adam + binary_crossentropy + AUC | Compiled model |
| 7    | Set up EarlyStopping, ModelCheckpoint, ReduceLROnPlateau | Training safeguards |
| 8    | Trained on real NIH data (no GAN) | `baseline_cnn_best.keras` |
| 9    | Plotted loss + AUC training curves | `baseline_training_curves.png` |
| 10–11| Evaluated on held-out test set, per-class AUC/F1 | `baseline_metrics.csv` |
| 12   | ROC curves per disease class | `baseline_roc_curves.png` |
| 13   | Visual prediction check on sample images | `baseline_sample_predictions.png` |
| 14   | Saved all metrics to CSV for Phase 5 comparison | `baseline_metrics.csv` |

### Key insight to note for your report:
Look at the per-class results. You should see that **Pneumonia** and **Cardiomegaly** — the minority classes identified in Phase 1 — have noticeably lower AUC and F1 than **Effusion** and **No Finding**. This is the class imbalance problem in action. Record these numbers carefully — they are your evidence that the GAN augmentation in Phase 3 is needed.

**Next step → Phase 3:** Build the GAN to generate synthetic minority-class X-rays, then retrain this same CNN on the augmented dataset and compare results.